# CatBoost Downstream Reference for FRC Compressive Strength


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
DATA_PATH = ROOT / "frc_compressive_strength.csv"
OUTPUT_DIR = ROOT / "outputs"
TARGET = "CS"
RANDOM_STATE = 42

FEATURES = [
    "C", "SF", "FA", "Fine_Agg", "Coarse_Agg", "W", "SP", "D",
    "W_C_ratio", "Total_Binder", "Total_Fiber", "SF_C_ratio",
    "W_Binder_ratio", "Fiber_Aspect_Ratio",
]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)

df = pd.read_csv(DATA_PATH)
x_train, x_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.2, random_state=RANDOM_STATE
)

model = CatBoostRegressor(
    iterations=1200,
    depth=4,
    learning_rate=0.1,
    l2_leaf_reg=3.0,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
)
model.fit(x_train, y_train)

train_pred = model.predict(x_train)
test_pred = model.predict(x_test)

metrics = pd.DataFrame([
    {"Metric": "R2", "Train": r2_score(y_train, train_pred), "Test": r2_score(y_test, test_pred)},
    {"Metric": "RMSE", "Train": rmse(y_train, train_pred), "Test": rmse(y_test, test_pred)},
    {"Metric": "MAE", "Train": mean_absolute_error(y_train, train_pred), "Test": mean_absolute_error(y_test, test_pred)},
    {"Metric": "MAPE", "Train": mape(y_train, train_pred), "Test": mape(y_test, test_pred)},
])

predictions = pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": test_pred})
predictions["Error"] = predictions["Actual"] - predictions["Predicted"]
predictions["Absolute_Error"] = predictions["Error"].abs()

feature_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.get_feature_importance(),
}).sort_values("Importance", ascending=False)

OUTPUT_DIR.mkdir(exist_ok=True)
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
metrics
